# From Waveform to Genre — Modeling & Training


**FMA dataset (citation)**  
Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017). *FMA: A Dataset for Music Analysis*. 18th ISMIR. PDF: https://arxiv.org/pdf/1612.01840.pdf

**License / data note:** FMA metadata is CC BY 4.0; audio files follow per-artist Creative Commons licenses. Consult the original dataset for terms of use.

---

### Project adaptation note
This notebook adapts the FMA data pipeline for the current workflow and interactive environments (Google Colab / Drive). Key adaptations include Colab/Drive path variables, idempotent download & extraction steps, automatic manifest generation for reproducibility, and helper utilities for checksums and seeding. Any reuse of original code or logic from the mdeff/fma project is indicated in the header and documented in the repository.


***
This notebook prepares and trains an audio-based genre classifier end-to-end. It configures the Colab environment and project paths, defines audio preprocessing and dataset utilities, builds a PyTorch dataset/loader that converts raw audio into normalized Mel-spectrogram inputs, and implements the model training loop with checkpointing and training-history logging. After training, it runs inference on the test split, saves per-track predictions and evaluation metrics (confusion matrix, classification report), and produces diagnostics and visualizations — including learning curves, latent-space projections (t-SNE/UMAP) with silhouette scores, class distribution analysis, and example spectrograms for misclassified tracks. Finally, the notebook records provenance metadata and synchronizes the processed data and outputs to Google Drive.

### Short comparison vs. original mdeff/fma
1. Focused on end-to-end model training in Colab (audio waveform → mel-spectrogram → CNN) rather than the original repo's feature-centric tooling.

2. Adds Colab/Drive integration, automated metadata download + SHA-1 verification, balanced per-genre sampling, and stratified train/val/test splits.

3. Implements data provenance saving and Drive sync (overwrites previous copy) for reproducibility.

4. Includes inference, evaluation, visualization (t-SNE/UMAP, confusion matrices, learning curves) and error-analysis pipelines not present in the original.

### Mount Drive and Configure Project Paths
This mounts Google Drive in Colab, sets up project and data directories, ensures the local data folder exists, changes the working directory to the project folder, and prints the local data path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive/waveform_analysis_outputs"
os.makedirs(LOCAL_DATA, exist_ok=True)
os.chdir(WORKDIR)
print("WORKDIR set, LOCAL_DATA:", LOCAL_DATA)

Sets up required libraries and a compact modeling configuration (audio paths and preprocessing params, random seed, training hyperparameters, and output file locations) for training an audio genre classifier.

In [ ]:
# standard libs for filesystem, archives, downloads, and utility operations
import os, zipfile, urllib.request, shutil, time, datetime
# data libraries for arrays and tabular data
import numpy as np, pandas as pd
# PyTorch for model definition and training
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# audio processing library used to load and convert audio to spectrograms
import librosa

# Modeling config (kept compact)
# URL to the small FMA audio archive (download source for audio files)
AUDIO_URL = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
# local path where the downloaded audio ZIP will be stored
AUDIO_ZIP_LOCAL = os.path.join(LOCAL_DATA, "fma_small.zip")
AUDIO_DIR = os.path.join(LOCAL_DATA, "audio")
# audio preprocessing parameters: sampling rate and clip length (seconds)
SAMPLE_RATE = 22050
CLIP_SECONDS = 10
# spectrogram parameters: number of mel bands, FFT window size, and hop length
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
# reproducibility: fixed random seed for sampling/splitting/initialization
SEED = 42
# training hyperparameters: batch size, learning rate, and number of epochs
BATCH_SIZE = 16
LR = 5e-4
EPOCHS = 1   # demo short training by default
# paths for saving model weights and training history (CSV)
MODEL_PATH = os.path.join(LOCAL_DATA, "genre_classifier_net.pt")
HISTORY_CSV = os.path.join(LOCAL_DATA, "training_history.csv")

### Automated Dataset Restoration from Google Drive

Description: This code automates the restoration of the workspace database by intelligently searching for the most recent copy of the data_storage folder or the train.csv file within Google Drive and copying it to the local Colab environment to avoid re-generating the data.

In [ ]:
# single-cell solution: find data_storage ONLY within waveform_analysis_outputs in Drive and copy to local WORKDIR
from pathlib import Path
import os, shutil, glob, sys

WORKDIR = "/content/waveform_genre_project"
LOCAL_DATA = os.path.join(WORKDIR, "data_storage")
DRIVE_ROOT = "/content/drive/MyDrive"
SEARCH_ROOT = os.path.join(DRIVE_ROOT, "waveform_analysis_outputs")  # search is limited to this directory

# 1) Mount Drive if needed (Colab)
if not os.path.exists("/content/drive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped or failed (not Colab?). Continue if Drive already mounted.")
else:
    # quick check whether MyDrive exists
    if not os.path.exists(DRIVE_ROOT):
        print("Warning: /content/drive/MyDrive not found. If you mounted Drive under a different path, adjust DRIVE_ROOT.")

# 2) Ensure WORKDIR exists and switch to it
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Working dir:", os.getcwd())

# 3) If local data already present and has train.csv -> done
local_train = os.path.join(LOCAL_DATA, "train.csv")
if os.path.exists(local_train):
    print("Local train.csv already present at", local_train)
    print("You can continue running the notebook.")
    pass

# 4) Search for data_storage folders ONLY under waveform_analysis_outputs (recursive)
candidates = glob.glob(os.path.join(SEARCH_ROOT, "**", "data_storage"), recursive=True)
candidates = [c for c in candidates if os.path.isdir(c)]
print("Found data_storage candidate folders in Drive (within waveform_analysis_outputs):", len(candidates))

def score_path(p):
    # score by number of files (bigger is better) and mtime
    try:
        n_files = sum(1 for _ in Path(p).rglob('*') if _.is_file())
        mtime = Path(p).stat().st_mtime
        return (n_files, mtime)
    except Exception:
        return (0, 0)

# 5) If candidates found -> pick best and copy
if candidates:
    scored = [(score_path(p), p) for p in candidates]
    scored.sort(reverse=True)  # prefer more files / newer
    best = scored[0][1]
    print("Selected candidate to copy:", best)
    # copytree with dirs_exist_ok (Py3.8+). If older, do safe fallback.
    try:
        shutil.copytree(best, LOCAL_DATA, dirs_exist_ok=True)
    except TypeError:
        # older shutil (unlikely) -> manual copy
        for root, dirs, files in os.walk(best):
            rel = os.path.relpath(root, best)
            target_dir = os.path.join(LOCAL_DATA, rel) if rel != "." else LOCAL_DATA
            os.makedirs(target_dir, exist_ok=True)
            for f in files:
                srcf = os.path.join(root, f)
                dstf = os.path.join(target_dir, f)
                if not os.path.exists(dstf):
                    shutil.copy2(srcf, dstf)
    print("Copy complete. Local data_storage now at:", LOCAL_DATA)
    if os.path.exists(local_train):
        print("train.csv found locally -> OK. You can re-run the cell that failed earlier.")
    else:
        print("Warning: train.csv not found inside the copied data_storage. List files in LOCAL_DATA root:")
        print(os.listdir(LOCAL_DATA)[:50])
    pass

# 6) If no data_storage folder: search for any train.csv ONLY under waveform_analysis_outputs
found_trains = glob.glob(os.path.join(SEARCH_ROOT, "**", "train.csv"), recursive=True)
if found_trains:
    # choose the first or best candidate (we pick the one with largest size)
    found_trains = sorted(found_trains, key=lambda p: os.path.getsize(p) if os.path.exists(p) else 0, reverse=True)
    train_path = found_trains[0]
    parent = os.path.dirname(train_path)
    print("Found train.csv at:", train_path)
    print("Copying its parent folder:", parent, "->", LOCAL_DATA)
    try:
        shutil.copytree(parent, LOCAL_DATA, dirs_exist_ok=True)
    except TypeError:
        for root, dirs, files in os.walk(parent):
            rel = os.path.relpath(root, parent)
            target_dir = os.path.join(LOCAL_DATA, rel) if rel != "." else LOCAL_DATA
            os.makedirs(target_dir, exist_ok=True)
            for f in files:
                srcf = os.path.join(root, f)
                dstf = os.path.join(target_dir, f)
                if not os.path.exists(dstf):
                    shutil.copy2(srcf, dstf)
    print("Copy complete. Check:", os.path.join(LOCAL_DATA, "train.csv"))
    if os.path.exists(local_train):
        print("train.csv local copy OK. You can re-run the failing cell.")
    else:
        print("train.csv still not found after copy; list LOCAL_DATA contents:", os.listdir(LOCAL_DATA)[:50])
    pass
else:
    # 7) Nothing found in Drive under waveform_analysis_outputs
    print("No data_storage folder and no train.csv found under", SEARCH_ROOT)